# SimCLR Pipeline — CIFAR-10

This notebook walks through the full SimCLR self-supervised learning pipeline:

| Step | Cell | Description |
|------|------|-------------|
| 1 | Setup | Imports, working directory, device |
| 2 | Config | Load hyperparameters from YAML |
| 3 | Dataset preview | Visualise raw + augmented CIFAR-10 samples |
| 4 | Model | Inspect SimCLR architecture |
| 5 | Pre-training | Train the SimCLR backbone (NT-Xent loss) |
| 6 | Pre-train plots | Loss curve |
| 7 | Linear evaluation | Freeze encoder, train linear classifier |
| 8 | Test evaluation | Evaluate on held-out test set |
| 9 | Linear eval plots | Loss, accuracy, per-class bar chart |
| 10 | Summary | Print final metrics |

> **Note:** Run cells top-to-bottom. Steps 5 and 7 are the long-running training cells.

---
## 1 · Setup

In [ ]:
import os, json, sys, time

# Ensure repo root is on sys.path so local modules resolve correctly
REPO_ROOT = os.path.abspath(".")          # adjust if you open the notebook from elsewhere
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torchvision import datasets, transforms

from utils.data_utils import Config, get_device, set_seed
from utils.simclr_utils import SimCLRDataset
from models.simclr import SimCLRModel

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

DEVICE = get_device()
print(f"Device: {DEVICE}")

---
## 2 · Configuration

Edit `CONFIG_PATH` or modify values in the cell below to change hyperparameters.

In [ ]:
CONFIG_PATH      = "config/simclr_config.yml"
CHECKPOINT_DIR   = "checkpoints"
PLOTS_DIR        = "plots"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

with open(CONFIG_PATH) as f:
    raw_cfg = yaml.safe_load(f)

config = Config(raw_cfg)

print("=== Loaded config ===")
print(f"  dataset      : {config.data.dataset}")
print(f"  batch_size   : {config.train.batch_size}")
print(f"  lr           : {config.train.lr}")
print(f"  n_epochs     : {config.train.n_epochs}")
print(f"  num_workers  : {config.train.num_workers}")
print(f"  proj_dim     : {config.network.proj_dim}")
print(f"  temperature  : {config.simclr.temperature}")
print(f"  weight_decay : {config.optimizer.weight_decay}")
print(f"  momentum     : {config.optimizer.momentum}")

---
## 3 · Dataset Preview

Visualise raw CIFAR-10 images alongside their two SimCLR augmented views to understand what the model learns from.

In [ ]:
CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

# Raw dataset (no transform) so we can show the PIL image
raw_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=None)

# SimCLR augmented wrapper
aug_dataset = SimCLRDataset(datasets.CIFAR10(root="./data", train=True, download=False, transform=None))

CIFAR_MEAN = np.array([0.4914, 0.4822, 0.4465])
CIFAR_STD  = np.array([0.2470, 0.2435, 0.2616])

def denorm(tensor):
    """Undo CIFAR-10 normalisation for display."""
    img = tensor.permute(1, 2, 0).numpy()
    img = img * CIFAR_STD + CIFAR_MEAN
    return np.clip(img, 0, 1)

N = 6   # number of examples to show
indices = [0, 1, 2, 3, 4, 5]

fig, axes = plt.subplots(3, N, figsize=(N * 2.2, 7))
fig.suptitle("Row 1: Original   |   Row 2: Augmented view 1   |   Row 3: Augmented view 2",
             fontsize=11, y=1.01)

for col, idx in enumerate(indices):
    pil_img, label = raw_dataset[idx]
    x1, x2 = aug_dataset[idx]

    axes[0, col].imshow(pil_img)
    axes[0, col].set_title(CIFAR10_CLASSES[label], fontsize=9)

    axes[1, col].imshow(denorm(x1))
    axes[2, col].imshow(denorm(x2))

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 4 · Model Architecture

In [ ]:
model = SimCLRModel(config)

total_params   = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
proj_params    = sum(p.numel() for p in model.projector.parameters())
cls_params     = sum(p.numel() for p in model.classifier.parameters())

print("=== SimCLR Model ===")
print(model)
print()
print(f"Total parameters   : {total_params:,}")
print(f"  Encoder (ResNet-18) : {encoder_params:,}")
print(f"  Projection MLP      : {proj_params:,}   (used during pre-training)")
print(f"  Linear classifier   : {cls_params:,}   (used during linear eval)")
print()
print("Forward pass (pretrain mode):")
dummy = torch.randn(4, 3, 32, 32)
model.linear_eval = False
h, z = model(dummy)
print(f"  Input  : {tuple(dummy.shape)}")
print(f"  h (representation) : {tuple(h.shape)}")
print(f"  z (projection)     : {tuple(z.shape)}")
print()
print("Forward pass (linear eval mode):")
model.linear_eval = True
h, logits = model(dummy)
print(f"  logits : {tuple(logits.shape)}")

del model   # free memory before training

---
## 5 · Pre-training (SimCLR — NT-Xent Loss)

**Skip this cell if you already have `checkpoints/simclr_pretrain2_cifar.pth`.**  
Training takes ~30 min on CPU for 20 epochs.

In [ ]:
PRETRAIN_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "simclr_pretrain2_cifar.pth")
PRETRAIN_METRICS    = os.path.join(CHECKPOINT_DIR, "pretrain_metrics.json")

if os.path.exists(PRETRAIN_CHECKPOINT):
    print(f"Checkpoint already exists at {PRETRAIN_CHECKPOINT}")
    print("Skipping pre-training. Delete the file and re-run to retrain.")
else:
    from train.train_simclr import SimCLRTrainer
    print("Starting SimCLR pre-training...")
    trainer = SimCLRTrainer(config, checkpoint_dir=CHECKPOINT_DIR, device=DEVICE)
    trainer.train()
    print("Pre-training complete.")

---
## 6 · Pre-training Visualisation

In [ ]:
PRETRAIN_METRICS = os.path.join(CHECKPOINT_DIR, "pretrain_metrics.json")

with open(PRETRAIN_METRICS) as f:
    pre_data = json.load(f)

losses = pre_data["epoch_losses"]
n      = len(losses)
epochs = list(range(1, n + 1))
step   = max(1, n // 10)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, losses, color="#2196F3", linewidth=2, marker="o", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("NT-Xent Loss (avg per epoch)")
ax.set_title(
    f"SimCLR Pre-training Loss — {pre_data.get('dataset', 'CIFAR-10')}\n"
    f"({n} epochs, {pre_data.get('total_time_s', 0):.0f}s total)"
)
ax.set_xlim(0.5, n + 0.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(step))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
ax.grid(True, alpha=0.3)

best_epoch = int(np.argmin(losses)) + 1
best_val   = min(losses)
y_range    = max(losses) - min(losses) or 1
ax.annotate(
    f"best: {best_val:.3f}",
    xy=(best_epoch, best_val),
    xytext=(best_epoch - max(1, n * 0.12), best_val + y_range * 0.1),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=9, color="gray",
)

plt.tight_layout()
save_path = os.path.join(PLOTS_DIR, "pretrain_loss.png")
fig.savefig(save_path, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

---
## 7 · Linear Evaluation

The encoder is loaded from checkpoint and **frozen**. Only the linear classifier head is trained.

In [ ]:
from train.train_linear import LinearEvalTrainer

print("Starting linear evaluation...")
linear_trainer = LinearEvalTrainer(config, checkpoint_dir=CHECKPOINT_DIR, device=DEVICE)
linear_trainer.train()
print("Linear evaluation training complete.")

---
## 8 · Test Set Evaluation

In [ ]:
linear_trainer.test()

---
## 9 · Linear Evaluation Visualisations

In [ ]:
# ── 9a: Loss + Accuracy curves ──────────────────────────────────────────────
LINEAR_METRICS = os.path.join(CHECKPOINT_DIR, "linear_metrics.json")

with open(LINEAR_METRICS) as f:
    lin_data = json.load(f)

lin_losses = lin_data["epoch_losses"]
lin_accs   = lin_data["epoch_accs"]
n_lin      = len(lin_losses)
lin_epochs = list(range(1, n_lin + 1))
lin_step   = max(1, n_lin // 10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    f"Linear Evaluation — {lin_data.get('dataset', 'CIFAR-10')}   "
    f"({n_lin} epochs, {lin_data.get('total_time_s', 0):.0f}s total)",
    fontsize=12,
)

# Loss
ax1.plot(lin_epochs, lin_losses, color="#E91E63", linewidth=2, marker="o", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss (avg per epoch)")
ax1.set_title("Training Loss")
ax1.set_xlim(0.5, n_lin + 0.5)
ax1.xaxis.set_major_locator(ticker.MultipleLocator(lin_step))
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(lin_epochs, lin_accs, color="#4CAF50", linewidth=2, marker="o", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Training Accuracy")
ax2.set_xlim(0.5, n_lin + 0.5)
ax2.xaxis.set_major_locator(ticker.MultipleLocator(lin_step))
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
ax2.grid(True, alpha=0.3)

best_acc_epoch = int(np.argmax(lin_accs)) + 1
best_acc_val   = max(lin_accs)
y_range_acc    = max(lin_accs) - min(lin_accs) or 1
ax2.annotate(
    f"best: {best_acc_val:.1f}%",
    xy=(best_acc_epoch, best_acc_val),
    xytext=(best_acc_epoch - max(1, n_lin * 0.12), best_acc_val - y_range_acc * 0.15),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=9, color="gray",
)

plt.tight_layout()
save_path = os.path.join(PLOTS_DIR, "linear_eval.png")
fig.savefig(save_path, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

In [ ]:
# ── 9b: Per-class test accuracy bar chart ───────────────────────────────────
TEST_RESULTS = os.path.join(CHECKPOINT_DIR, "test_results.json")

with open(TEST_RESULTS) as f:
    test_data = json.load(f)

per_class = test_data["per_class_accuracy"]
avg_acc   = test_data["avg_accuracy"]
classes   = list(per_class.keys())
values    = [per_class[c] or 0.0 for c in classes]
colors    = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(classes, values, color=colors[:len(classes)], edgecolor="white", linewidth=0.6)
ax.axhline(avg_acc, color="black", linestyle="--", linewidth=1.2,
           label=f"Average: {avg_acc:.1f}%")
ax.legend(fontsize=9)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}", ha="center", va="bottom", fontsize=8.5)

ax.set_xlabel("Class")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-class Test Accuracy — CIFAR-10")
ax.set_ylim(0, max(values) * 1.15)
plt.xticks(rotation=20, ha="right")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
save_path = os.path.join(PLOTS_DIR, "test_per_class_accuracy.png")
fig.savefig(save_path, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

In [ ]:
# ── 9c: Pre-train vs linear eval loss (dual axis) ───────────────────────────
PRETRAIN_METRICS = os.path.join(CHECKPOINT_DIR, "pretrain_metrics.json")

with open(PRETRAIN_METRICS) as f:
    pre_data = json.load(f)

pre_epochs = list(range(1, len(pre_data["epoch_losses"]) + 1))
lin_epochs = list(range(1, len(lin_data["epoch_losses"]) + 1))
max_n      = max(len(pre_epochs), len(lin_epochs))
step       = max(1, max_n // 10)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(pre_epochs, pre_data["epoch_losses"], color="#2196F3", linewidth=2,
        label="Pre-training (NT-Xent)", marker="o", markersize=3)
ax2 = ax.twinx()
ax2.plot(lin_epochs, lin_data["epoch_losses"], color="#E91E63", linewidth=2,
         label="Linear Eval (CE)", marker="s", markersize=3)

ax.set_xlabel("Epoch")
ax.set_ylabel("NT-Xent Loss (avg per epoch)", color="#2196F3")
ax2.set_ylabel("Cross-Entropy Loss (avg per epoch)", color="#E91E63")
ax.tick_params(axis="y", labelcolor="#2196F3")
ax2.tick_params(axis="y", labelcolor="#E91E63")
ax2.spines["right"].set_visible(True)

ax.xaxis.set_major_locator(ticker.MultipleLocator(step))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x)}"))
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)
ax.set_title("Pre-training vs. Linear Eval Loss")

plt.tight_layout()
save_path = os.path.join(PLOTS_DIR, "combined_loss.png")
fig.savefig(save_path, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

---
## 10 · Summary

In [ ]:
PRETRAIN_METRICS = os.path.join(CHECKPOINT_DIR, "pretrain_metrics.json")
LINEAR_METRICS   = os.path.join(CHECKPOINT_DIR, "linear_metrics.json")
TEST_RESULTS     = os.path.join(CHECKPOINT_DIR, "test_results.json")

with open(PRETRAIN_METRICS) as f: pre  = json.load(f)
with open(LINEAR_METRICS)   as f: lin  = json.load(f)
with open(TEST_RESULTS)     as f: test = json.load(f)

print("╔══════════════════════════════════════════════╗")
print("║           SimCLR Pipeline Summary            ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  Dataset            : {pre.get('dataset','CIFAR-10'):<22} ║")
print("╠══════════════════════════════════════════════╣")
print("║  PRE-TRAINING                                ║")
print(f"║    Epochs           : {pre['n_epochs']:<22} ║")
print(f"║    Final loss       : {pre['epoch_losses'][-1]:<22.4f} ║")
print(f"║    Best loss        : {min(pre['epoch_losses']):<22.4f} ║")
print(f"║    Total time       : {pre['total_time_s']:<22.1f} ║")
print("╠══════════════════════════════════════════════╣")
print("║  LINEAR EVALUATION                           ║")
print(f"║    Epochs           : {lin['n_epochs']:<22} ║")
print(f"║    Final train acc  : {lin['epoch_accs'][-1]:<22.2f} ║")
print(f"║    Best train acc   : {max(lin['epoch_accs']):<22.2f} ║")
print(f"║    Total time       : {lin['total_time_s']:<22.1f} ║")
print("╠══════════════════════════════════════════════╣")
print("║  TEST SET                                    ║")
print(f"║    Average accuracy : {test['avg_accuracy']:<21.2f}% ║")
print(f"║    Average loss     : {test['avg_loss']:<22.4f} ║")
print("╠══════════════════════════════════════════════╣")
print("║  Per-class test accuracy:                    ║")
for cls, acc in test["per_class_accuracy"].items():
    bar = "█" * int((acc or 0) / 5)
    print(f"║    {cls:<12}: {acc:>5.1f}%  {bar:<20} ║")
print("╚══════════════════════════════════════════════╝")